# KLWX storm-cluster Kalman tracker

This notebook tracks a storm **cluster** rather than demanding that individual thresholded cells retain the same identity. Nearby cells are merged into one cluster; a constant-velocity Kalman filter estimates its position and velocity, then provides forecast uncertainty.

## Case and method

We use four KLWX Level II scans during the June 5, 2024 Poolesville-area severe-weather event. The initial seed is near Poolesville relative to the Sterling radar. Later observations are the nearest plausible storm cluster to the filter prediction. This is a reproducible baseline, not a tornado detector.

In [ ]:
%pip install -e "../.[nexrad]"

In [ ]:
from pathlib import Path
import sys
import boto3
from botocore import UNSIGNED
from botocore.config import Config
import matplotlib.pyplot as plt
import numpy as np

repo = Path.cwd().parent
sys.path.insert(0, str(repo / 'src'))
from storm_motion.impact import equivalent_cluster_radius_m, local_xy_from_lat_lon, point_impact_probability
from storm_motion.mosaic import max_reflectivity_mosaic
from storm_motion.nexrad import load_lowest_reflectivity_sweep
from storm_motion.tracking import ConstantVelocityKalmanTracker, RadarFrame, cluster_cells, detect_cells

bucket = 'unidata-nexrad-level2'
keys = [
    '2024/06/05/KLWX/KLWX20240605_230437_V06',
    '2024/06/05/KLWX/KLWX20240605_231039_V06',
    '2024/06/05/KLWX/KLWX20240605_231643_V06',
    '2024/06/05/KLWX/KLWX20240605_232308_V06',
]
data_dir = repo / 'data' / 'klwx_20240605'
data_dir.mkdir(parents=True, exist_ok=True)
s3 = boto3.client('s3', region_name='us-east-1', config=Config(signature_version=UNSIGNED) )
paths = []
for key in keys:
    path = data_dir / Path(key).name
    if not path.exists():
        s3.download_file(bucket, key, str(path))
    paths.append(path)
frames = [load_lowest_reflectivity_sweep(path) for path in paths]
print(*[frame.observed_at.isoformat() for frame in frames], sep='\n')

In [ ]:
# Limit detection to the Poolesville-area window and form clusters from nearby cells.
roi_km = dict(xmin=-35, xmax=35, ymin=-10, ymax=50)
def in_roi(frame):
    x_km, y_km = frame.x_m / 1_000, frame.y_m / 1_000
    inside = (x_km >= roi_km['xmin']) & (x_km <= roi_km['xmax']) & (y_km >= roi_km['ymin']) & (y_km <= roi_km['ymax'])
    return RadarFrame(frame.observed_at, np.where(inside, frame.reflectivity_dbz, np.nan), frame.x_m, frame.y_m, frame.radar_latitude_deg, frame.radar_longitude_deg)

def clusters_for(frame):
    # The same storm can weaken below a fixed reflectivity cutoff between scans.
    for threshold_dbz, min_gates in [(35, 20), (30, 12), (25, 8)]:
        clusters = cluster_cells(detect_cells(frame, threshold_dbz=threshold_dbz, min_gates=min_gates), linkage_distance_m=15_000)
        if clusters:
            return clusters, threshold_dbz, min_gates
    raise RuntimeError('No storm clusters found even at 25 dBZ; widen the region of interest.')

roi_frames = [in_roi(frame) for frame in frames]
# Keep only the one unmasked source frame needed later for the multi-radar mosaic.
final_source_frame = frames[-1]
del frames
cluster_results = [clusters_for(frame) for frame in roi_frames]
cluster_sets = [result[0] for result in cluster_results]
for frame, (clusters, threshold_dbz, min_gates) in zip(roi_frames, cluster_results):
    print(f'{frame.observed_at:%H:%M:%S UTC}: {len(clusters)} clusters at {threshold_dbz:.0f} dBZ / {min_gates} gates')

In [ ]:
# Seed near Poolesville (roughly 0 km east, 16 km north of KLWX), then let the Kalman
# prediction choose the closest cluster in each later scan.
seed_x_m, seed_y_m = 0, 16_000
first_cluster = min(cluster_sets[0], key=lambda cluster: np.hypot(cluster.centroid_x_m - seed_x_m, cluster.centroid_y_m - seed_y_m))
tracker = ConstantVelocityKalmanTracker(measurement_std_m=4_000, acceleration_std_mps2=0.01)
tracker.update(roi_frames[0].observed_at, first_cluster.centroid_x_m, first_cluster.centroid_y_m)
history = [(roi_frames[0], first_cluster, None, tracker.estimate)]

for frame, clusters in zip(roi_frames[1:], cluster_sets[1:]):
    prediction = tracker.predict(frame.observed_at)
    observed = min(clusters, key=lambda cluster: np.hypot(cluster.centroid_x_m - prediction.x_m, cluster.centroid_y_m - prediction.y_m))
    estimate = tracker.update(frame.observed_at, observed.centroid_x_m, observed.centroid_y_m)
    history.append((frame, observed, prediction, estimate))

for frame, cluster, prediction, estimate in history:
    error = 0 if prediction is None else np.hypot(cluster.centroid_x_m - prediction.x_m, cluster.centroid_y_m - prediction.y_m)
    print(f'{frame.observed_at:%H:%M:%S UTC}: cluster {cluster.cluster_id}, prediction error {error / 1_000:.2f} km, filtered speed {np.hypot(estimate.velocity_x_mps, estimate.velocity_y_mps):.1f} m/s')

In [ ]:
# Yellow = forecast before seeing this scan; cyan = observed cluster; white = Kalman-filtered position.
fig, axes = plt.subplots(2, 2, figsize=(13, 12), sharex=True, sharey=True)
for ax, (frame, cluster, prediction, estimate) in zip(axes.flat, history):
    image = ax.pcolormesh(frame.x_m / 1_000, frame.y_m / 1_000, frame.reflectivity_dbz, cmap='turbo', vmin=0, vmax=65)
    if prediction is not None:
        ax.scatter(prediction.x_m / 1_000, prediction.y_m / 1_000, s=115, c='#ffdf00', edgecolors='black', zorder=4, label='predicted')
        circle = plt.Circle((prediction.x_m / 1_000, prediction.y_m / 1_000), prediction.position_std_m / 1_000, facecolor='#ffdf00', edgecolor='white', linewidth=2.5, linestyle='--', alpha=.35)
        ax.add_patch(circle)
    ax.scatter(cluster.centroid_x_m / 1_000, cluster.centroid_y_m / 1_000, s=115, c='#00e5ff', edgecolors='black', zorder=4, label='observed cluster')
    ax.scatter(estimate.x_m / 1_000, estimate.y_m / 1_000, s=185, facecolors='none', edgecolors='white', linewidths=2, zorder=5, label='filtered')
    ax.set(title=f'{frame.observed_at:%H:%M:%S UTC}', xlabel='km east of KLWX', ylabel='km north of KLWX', xlim=(roi_km['xmin'], roi_km['xmax']), ylim=(roi_km['ymin'], roi_km['ymax']), aspect='equal')
    ax.legend(loc='upper left')
fig.colorbar(image, ax=axes.ravel().tolist(), label='reflectivity (dBZ)'); plt.show(); plt.close(fig)

In [ ]:
# Forecast from the final filtered state. The yellow circle is one standard deviation; it grows with lead time.
last_observation_time = tracker.estimate.observed_at
forecast_minutes = [30, 60, 240, 480]
def lead_label(minutes):
    return f'+{minutes // 60} hour' if minutes % 60 == 0 else f'+{minutes} min'
forecasts = []
for minutes in forecast_minutes:
    forecast = tracker.predict(last_observation_time + np.timedelta64(minutes, 'm').astype('timedelta64[s]').astype(object))
    forecasts.append((minutes, forecast))
    print(f'{lead_label(minutes)}: ({forecast.x_m / 1_000:.1f}, {forecast.y_m / 1_000:.1f}) km; ±{forecast.position_std_m / 1_000:.1f} km (1σ)')

final_frame = roi_frames[-1]
fig, ax = plt.subplots(figsize=(9, 7))
image = ax.pcolormesh(final_frame.x_m / 1_000, final_frame.y_m / 1_000, final_frame.reflectivity_dbz, cmap='turbo', vmin=0, vmax=65)
for minutes, forecast in forecasts:
    ax.scatter(forecast.x_m / 1_000, forecast.y_m / 1_000, s=105, c='#ffdf00', edgecolors='black', zorder=4)
    ax.add_patch(plt.Circle((forecast.x_m / 1_000, forecast.y_m / 1_000), forecast.position_std_m / 1_000, facecolor='#ffdf00', edgecolor='white', linewidth=2.5, linestyle='--', alpha=.35))
    ax.annotate(f'{lead_label(minutes)}\n±{forecast.position_std_m / 1_000:.1f} km', (forecast.x_m / 1_000, forecast.y_m / 1_000), xytext=(7, 7), textcoords='offset points', color='white', weight='bold')
ax.set(title='Kalman storm-cluster forecast', xlabel='km east of KLWX', ylabel='km north of KLWX', xlim=(roi_km['xmin'], roi_km['xmax']), ylim=(roi_km['ymin'], roi_km['ymax']), aspect='equal')
fig.colorbar(image, ax=ax, label='reflectivity (dBZ)'); plt.show(); plt.close(fig)

## McLean office impact baseline

This converts the moving cluster into a **point-impact proxy** for Booz Allen's McLean office at 8283 Greensboro Drive (38.9224° N, 77.2321° W). Each Monte Carlo sample shifts the cluster centre using the Kalman position uncertainty and asks whether a same-area circular reflectivity footprint covers the office. It reports a chance of thresholded radar reflectivity reaching the point—not a probability of severe weather, damage, or an official warning.

In [ ]:
# Office coordinate is a provisional geocode for 8283 Greensboro Drive, McLean, VA.
office_latitude_deg, office_longitude_deg = 38.922403, -77.232125
if final_frame.radar_latitude_deg is None or final_frame.radar_longitude_deg is None:
    raise RuntimeError('The NEXRAD loader did not provide the radar location.')
office_x_m, office_y_m = local_xy_from_lat_lon(
    office_latitude_deg, office_longitude_deg,
    final_frame.radar_latitude_deg, final_frame.radar_longitude_deg,
)

# Approximate a polar radar gate area at the tracked cluster's range.
final_cluster = history[-1][1]
range_step_m = np.nanmedian(np.diff(np.hypot(final_frame.x_m[0], final_frame.y_m[0])))
azimuths_rad = np.unwrap(np.arctan2(final_frame.x_m[:, 0], final_frame.y_m[:, 0]))
azimuth_step_rad = np.nanmedian(np.abs(np.diff(azimuths_rad)))
cluster_range_m = np.hypot(final_cluster.centroid_x_m, final_cluster.centroid_y_m)
gate_area_m2 = cluster_range_m * range_step_m * azimuth_step_rad
footprint_radius_m = equivalent_cluster_radius_m(final_cluster, gate_area_m2)
print(f'McLean office: ({office_x_m / 1_000:.1f}, {office_y_m / 1_000:.1f}) km from KLWX')
print(f'Equivalent {cluster_results[-1][1]:.0f} dBZ cluster footprint radius: {footprint_radius_m / 1_000:.1f} km')

In [ ]:
impact_forecasts = []
for minutes, forecast in forecasts:
    impact = point_impact_probability(
        forecast, office_x_m, office_y_m, footprint_radius_m, random_seed=minutes,
    )
    impact_forecasts.append((minutes, forecast, impact))
    print(
        f'+{minutes:2d} min: {impact.probability:.1%} chance this thresholded cluster covers McLean '
        f'(forecast centre is {impact.target_distance_m / 1_000:.1f} km away)'
    )

fig, ax = plt.subplots(figsize=(9, 7))
image = ax.pcolormesh(final_frame.x_m / 1_000, final_frame.y_m / 1_000, final_frame.reflectivity_dbz, cmap='turbo', vmin=0, vmax=65)
ax.scatter(office_x_m / 1_000, office_y_m / 1_000, marker='*', s=280, c='#ff4d4d', edgecolors='white', linewidths=1.2, zorder=7, label='McLean office')
for minutes, forecast, impact in impact_forecasts:
    ax.scatter(forecast.x_m / 1_000, forecast.y_m / 1_000, s=105, c='#ffdf00', edgecolors='black', zorder=5)
    ax.add_patch(plt.Circle((forecast.x_m / 1_000, forecast.y_m / 1_000), footprint_radius_m / 1_000, fill=False, edgecolor='#ffdf00', linewidth=2, zorder=4))
    ax.add_patch(plt.Circle((forecast.x_m / 1_000, forecast.y_m / 1_000), forecast.position_std_m / 1_000, fill=False, edgecolor='white', linewidth=2.5, linestyle='--', alpha=.8, zorder=3))
    ax.annotate(f'{lead_label(minutes)}: {impact.probability:.0%}', (forecast.x_m / 1_000, forecast.y_m / 1_000), xytext=(7, 7), textcoords='offset points', color='white', weight='bold')
ax.set(title='McLean point-impact baseline: reflectivity-cluster coverage', xlabel='km east of KLWX', ylabel='km north of KLWX', xlim=(roi_km['xmin'], roi_km['xmax']), ylim=(roi_km['ymin'], roi_km['ymax']), aspect='equal')
ax.legend(loc='upper left'); fig.colorbar(image, ax=ax, label='reflectivity (dBZ)'); plt.show(); plt.close(fig)
print('Interpretation: yellow circle = assumed current storm footprint; white dashed circle = 1σ centre uncertainty.')

## McLean-centered regional radar composite

The tracker still uses KLWX reflectivity for this case. This regional view recenters the product on McLean and shows the overlapping operational coverage of KLWX (northwest), KDOX/Dover (east), and KAKQ/Wakefield (south). The translucent rings are coverage context—not a fused multi-radar reflectivity field yet.

In [ ]:
# Coordinates: KLWX comes from the Level II file; KDOX and KAKQ are WSR-88D site locations.
radar_sites = [
    ('KLWX · Sterling', final_frame.radar_latitude_deg, final_frame.radar_longitude_deg, '#00e5ff'),
    ('KDOX · Dover', 38.826, -75.440, '#7ee787'),
    ('KAKQ · Wakefield', 36.983, -77.007, '#c792ea'),
]

fig, ax = plt.subplots(figsize=(11, 9))
# Translate the real KLWX sweep into McLean-centered coordinates.
image = ax.pcolormesh(
    (final_frame.x_m - office_x_m) / 1_000, (final_frame.y_m - office_y_m) / 1_000,
    final_frame.reflectivity_dbz, cmap='turbo', vmin=0, vmax=65, alpha=.78,
)
coverage_radius_km = 230  # useful low-level operational coverage context, not a hard limit
for name, latitude, longitude, color in radar_sites:
    radar_x_m, radar_y_m = local_xy_from_lat_lon(latitude, longitude, office_latitude_deg, office_longitude_deg)
    ax.add_patch(plt.Circle((radar_x_m / 1_000, radar_y_m / 1_000), coverage_radius_km, facecolor=color, edgecolor=color, alpha=.055, linewidth=1.6, zorder=1))
    ax.scatter(radar_x_m / 1_000, radar_y_m / 1_000, marker='^', s=115, c=color, edgecolors='black', zorder=6)
    ax.annotate(name, (radar_x_m / 1_000, radar_y_m / 1_000), xytext=(6, 6), textcoords='offset points', color=color, weight='bold')

ax.scatter(0, 0, marker='*', s=360, c='#ff4d4d', edgecolors='white', linewidths=1.2, zorder=8, label='McLean office')
for minutes, forecast, impact in impact_forecasts:
    forecast_x_km, forecast_y_km = (forecast.x_m - office_x_m) / 1_000, (forecast.y_m - office_y_m) / 1_000
    ax.scatter(forecast_x_km, forecast_y_km, s=115, c='#ffdf00', edgecolors='black', zorder=7)
    ax.add_patch(plt.Circle((forecast_x_km, forecast_y_km), footprint_radius_m / 1_000, fill=False, edgecolor='#ffdf00', linewidth=2.2, zorder=6))
    ax.add_patch(plt.Circle((forecast_x_km, forecast_y_km), forecast.position_std_m / 1_000, fill=False, edgecolor='white', linewidth=2.5, linestyle='--', alpha=.9, zorder=5))
    ax.annotate(f'{lead_label(minutes)}\n{impact.probability:.0%}', (forecast_x_km, forecast_y_km), xytext=(7, 7), textcoords='offset points', color='white', weight='bold')

ax.axhline(0, color='white', alpha=.25, linewidth=.8); ax.axvline(0, color='white', alpha=.25, linewidth=.8)
ax.set(
    title='McLean-centered radar coverage and KLWX storm forecast',
    xlabel='km east (+) / west (−) of McLean', ylabel='km north (+) / south (−) of McLean',
    xlim=(-150, 275), ylim=(-300, 150), aspect='equal',
)
ax.legend(loc='upper left'); fig.colorbar(image, ax=ax, label='KLWX reflectivity (dBZ)'); plt.show(); plt.close(fig)
print('The next cell builds the actual time-nearest, max-dBZ reflectivity mosaic on this grid.')

## Fused KLWX + KDOX + KAKQ reflectivity mosaic

This downloads the closest scans to the final KLWX observation (all within about three minutes), translates each sweep to the common McLean-centered grid, and keeps the maximum valid dBZ in every 1 km grid cell. This is a deliberately simple fusion rule: it visibly combines the real scans but does not yet correct for beam height, time offsets, attenuation, or quality differences.

In [ ]:
mosaic_keys = [
    '2024/06/05/KDOX/KDOX20240605_232114_V06',  # 23:21:14 UTC
    '2024/06/05/KAKQ/KAKQ20240605_232047_V06',  # 23:20:47 UTC
]
mosaic_paths = []
for key in mosaic_keys:
    path = data_dir / Path(key).name
    if not path.exists():
        s3.download_file(bucket, key, str(path))
    mosaic_paths.append(path)

mosaic_frames = [final_source_frame, *(load_lowest_reflectivity_sweep(path) for path in mosaic_paths)]
for frame in mosaic_frames:
    offset_minutes = (frame.observed_at - frames[-1].observed_at).total_seconds() / 60
    print(f'{frame.observed_at:%H:%M:%S UTC}: {offset_minutes:+.1f} min from KLWX final scan')

mosaic_x_m = np.arange(-150_000, 276_000, 1_000.0)
mosaic_y_m = np.arange(-300_000, 151_000, 1_000.0)
mosaic_dbz = max_reflectivity_mosaic(
    mosaic_frames, origin_latitude_deg=office_latitude_deg, origin_longitude_deg=office_longitude_deg,
    x_m=mosaic_x_m, y_m=mosaic_y_m,
)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 9))
image = ax.pcolormesh(mosaic_x_m / 1_000, mosaic_y_m / 1_000, mosaic_dbz, cmap='turbo', vmin=0, vmax=65, shading='nearest')
for name, latitude, longitude, color in radar_sites:
    radar_x_m, radar_y_m = local_xy_from_lat_lon(latitude, longitude, office_latitude_deg, office_longitude_deg)
    ax.scatter(radar_x_m / 1_000, radar_y_m / 1_000, marker='^', s=115, c=color, edgecolors='black', zorder=6)
    ax.annotate(name, (radar_x_m / 1_000, radar_y_m / 1_000), xytext=(6, 6), textcoords='offset points', color=color, weight='bold')
ax.scatter(0, 0, marker='*', s=360, c='#ff4d4d', edgecolors='white', linewidths=1.2, zorder=8, label='McLean office')
for minutes, forecast, impact in impact_forecasts:
    forecast_x_km, forecast_y_km = (forecast.x_m - office_x_m) / 1_000, (forecast.y_m - office_y_m) / 1_000
    ax.scatter(forecast_x_km, forecast_y_km, s=115, c='#ffdf00', edgecolors='black', zorder=7)
    ax.add_patch(plt.Circle((forecast_x_km, forecast_y_km), footprint_radius_m / 1_000, fill=False, edgecolor='#ffdf00', linewidth=2.2, zorder=6))
    ax.add_patch(plt.Circle((forecast_x_km, forecast_y_km), forecast.position_std_m / 1_000, fill=False, edgecolor='white', linewidth=2.5, linestyle='--', alpha=.9, zorder=5))
    ax.annotate(f'{lead_label(minutes)}\n{impact.probability:.0%}', (forecast_x_km, forecast_y_km), xytext=(7, 7), textcoords='offset points', color='white', weight='bold')
ax.axhline(0, color='white', alpha=.25, linewidth=.8); ax.axvline(0, color='white', alpha=.25, linewidth=.8)
ax.set(title='Fused KLWX + KDOX + KAKQ reflectivity mosaic, centered on McLean', xlabel='km east (+) / west (−) of McLean', ylabel='km north (+) / south (−) of McLean', xlim=(-150, 275), ylim=(-300, 150), aspect='equal')
ax.legend(loc='upper left'); fig.colorbar(image, ax=ax, label='maximum observed reflectivity (dBZ)'); plt.show(); plt.close(fig)
print('Fusion rule: maximum valid reflectivity per 1 km grid cell. The storm tracker is still KLWX-only in this phase.')

## McLean decision view: 25 km around the office

This is the same fused three-radar mosaic, cropped to a 50 × 50 km area (±25 km) centered on the McLean office. It is the focused decision view for the 30-minute and 1-hour forecasts. The 4-hour and 8-hour results are printed below as exploratory constant-velocity projections; they are intentionally not treated as operational nowcasts.

In [ ]:
local_radius_km = 25
local_resolution_m = 250  # Re-fuse original sweeps at NEXRAD range-gate scale; do not enlarge the 1 km overview.
local_x_m = np.arange(-local_radius_km * 1_000, (local_radius_km * 1_000) + local_resolution_m, local_resolution_m)
local_y_m = np.arange(-local_radius_km * 1_000, (local_radius_km * 1_000) + local_resolution_m, local_resolution_m)
local_mosaic_dbz = max_reflectivity_mosaic(
    mosaic_frames, origin_latitude_deg=office_latitude_deg, origin_longitude_deg=office_longitude_deg,
    x_m=local_x_m, y_m=local_y_m,
)
fig, ax = plt.subplots(figsize=(10, 9))
image = ax.pcolormesh(local_x_m / 1_000, local_y_m / 1_000, local_mosaic_dbz, cmap='turbo', vmin=0, vmax=65, shading='nearest')
ax.scatter(0, 0, marker='*', s=420, c='#ff4d4d', edgecolors='white', linewidths=1.4, zorder=9, label='Booz Allen · McLean')
for minutes, forecast, impact in impact_forecasts:
    forecast_x_km, forecast_y_km = (forecast.x_m - office_x_m) / 1_000, (forecast.y_m - office_y_m) / 1_000
    ax.scatter(forecast_x_km, forecast_y_km, s=135, c='#ffdf00', edgecolors='black', zorder=8)
    ax.add_patch(plt.Circle((forecast_x_km, forecast_y_km), footprint_radius_m / 1_000, fill=False, edgecolor='#ffdf00', linewidth=2.5, zorder=7))
    ax.add_patch(plt.Circle((forecast_x_km, forecast_y_km), forecast.position_std_m / 1_000, fill=False, edgecolor='white', linewidth=3, linestyle='--', alpha=.95, zorder=6))
    ax.annotate(f'{lead_label(minutes)} · {impact.probability:.0%}', (forecast_x_km, forecast_y_km), xytext=(8, 8), textcoords='offset points', color='white', weight='bold', fontsize=10)
ax.axhline(0, color='white', alpha=.3, linewidth=.9); ax.axvline(0, color='white', alpha=.3, linewidth=.9)
ax.set(
    title='McLean office: fused radar nowcast decision view',
    xlabel='km east (+) / west (−) of office', ylabel='km north (+) / south (−) of office',
    xlim=(-local_radius_km, local_radius_km), ylim=(-local_radius_km, local_radius_km), aspect='equal',
)
ax.legend(loc='upper left'); fig.colorbar(image, ax=ax, label='maximum observed reflectivity (dBZ)'); plt.show(); plt.close(fig)
print('McLEAN OFFICE IMPACT OUTLOOK')
for minutes, forecast, impact in impact_forecasts:
    confidence = 'baseline nowcast' if minutes <= 60 else 'exploratory constant-velocity projection'
    print(f'  {lead_label(minutes):>8}: {impact.probability:5.1%} modeled chance the tracked reflectivity cluster covers the office ({confidence})')
print('This is not yet a probability of severe weather or damage; that requires historical severe-weather labels and model validation.')
print('Rendered from all three original sweeps at 250 m cells; it is not an enlarged 1 km overview. Yellow outline: footprint. White dashed: 1σ centre uncertainty.')